# pydme — 1+1-D Gaussian Diffusion (self-contained Colab)

**What this notebook does**

1. Embeds the full `pydme` package source (no external install needed).
2. Generates a 1+1-D Gaussian diffusion dataset:
   `∂u/∂t = D ∂²u/∂x²`, solved analytically.
3. Fits **TMI (core)** — discovers spatial/temporal latents purely from data.
4. Builds **Galerkin weight matrices** (W, DW) for derivative-free dynamics.
5. Fits **TMI + dynamics** — additionally discovers the ODE `dz/dt ≈ ξ·θ(z)`
   using sparse regression, **without ever differentiating z**.

---

## 0 · Setup

Install PyTorch + NumPy (already present in Colab), then write the `pydme` source.

In [ ]:
# PyTorch and NumPy are pre-installed in Colab.
# Uncomment the line below only if running outside Colab.
# !pip install torch numpy -q
import sys, os
os.makedirs("pydme", exist_ok=True)

In [ ]:
%%writefile pydme/losses.py
"""Loss functions for pydme models."""

import torch


def kld_loss(
    p: torch.Tensor,
    q: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """KL divergence D_KL(p_norm || q_norm), summed over all elements.

    Both p and q are column-normalized before computing the divergence so
    that each column is treated as a discrete probability distribution.

    Supports 2D data (a, alpha) and 3D data (a, alpha, T), where for 3D
    the KLD is averaged over the trial dimension.

    Args:
        p: Target matrix.
        q: Model reconstruction, same shape as p.
        eps: Small value for numerical stability.

    Returns:
        Scalar KLD loss.
    """
    if p.ndim == 2:
        pn = p / (p.sum(dim=0, keepdim=True) + eps)
        qn = q / (q.sum(dim=0, keepdim=True) + eps)
        return (pn * torch.log((pn + eps) / (qn + eps))).sum()

    elif p.ndim == 3:
        n = p.shape[2]
        total = p.new_zeros(1).squeeze()
        for i in range(n):
            total = total + kld_loss(p[:, :, i], q[:, :, i], eps=eps)
        return total / n

    else:
        raise ValueError(f"p must be 2D or 3D, got {p.ndim}D")


In [ ]:
%%writefile pydme/library.py
"""Polynomial library construction for SINDy-style dynamics."""

import torch


def poly_library(z: torch.Tensor, N: torch.Tensor) -> torch.Tensor:
    """Build a polynomial feature library from latent variables z.

    Each library term l is the product z[:, 0]^N[l,0] * z[:, 1]^N[l,1] * ...

    Args:
        z: Latent variables of shape (T, K) or (T, K, trials).
        N: Exponent matrix of shape (L, K), where L is the number of library
           terms and K is the latent dimensionality.

    Returns:
        theta: Library matrix of shape (T, L) or (T, L, trials).
    """
    if N.device != z.device:
        N = N.to(z.device)

    if z.ndim == 2:
        T, K = z.shape
        L = N.shape[0]
        # Build each column separately to avoid in-place ops (required for autograd)
        cols = []
        for l in range(L):
            col = z.new_ones(T)
            for s in range(K):
                if N[l, s] != 0:
                    col = col * z[:, s] ** N[l, s]
            cols.append(col)
        return torch.stack(cols, dim=1)

    elif z.ndim == 3:
        T, K, trials = z.shape
        L = N.shape[0]
        cols = []
        for l in range(L):
            slices = []
            for j in range(trials):
                col = z.new_ones(T)
                for s in range(K):
                    if N[l, s] != 0:
                        col = col * z[:, s, j] ** N[l, s]
                slices.append(col)
            cols.append(torch.stack(slices, dim=1))   # (T, trials)
        return torch.stack(cols, dim=1)   # (T, L, trials)

    else:
        raise ValueError(f"z must be 2D or 3D, got {z.ndim}D")


In [ ]:
%%writefile pydme/sparse.py
"""Sparse regression with information-criterion model selection.

Implements Adaptive Sparse Regression (AdSR) and its ensemble variant
(EnAdSR), based on iterative thresholding with least-squares regression
and SLIC / AICc / BIC scoring.
"""

from __future__ import annotations

import math
from typing import Literal

import numpy as np
import torch


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------


def _lstsq_col(A: torch.Tensor, b: torch.Tensor, ridge: float) -> torch.Tensor:
    """Solve (A^T A + ridge * I) x = A^T b for x."""
    if ridge == 0.0:
        return torch.linalg.lstsq(A, b).solution
    lhs = A.T @ A + ridge * torch.eye(A.shape[1], dtype=A.dtype, device=A.device)
    rhs = A.T @ b
    return torch.linalg.solve(lhs, rhs)


def _update_active_coefs(
    theta: torch.Tensor,
    dz: torch.Tensor,
    xi: torch.Tensor,
    active: torch.Tensor,
    ridge: float = 0.0,
) -> torch.Tensor:
    """Re-fit coefficients for the active (nonzero) terms only.

    Args:
        theta: Library matrix (n_obs, n_terms).
        dz: Target matrix (n_obs, n_states).
        xi: Current coefficient matrix (n_terms, n_states).
        active: Boolean mask of active terms (n_terms, n_states).
        ridge: Ridge penalty.

    Returns:
        Updated coefficient matrix with inactive terms zeroed out.
    """
    xi = xi * active.to(xi.dtype)
    n_state = dz.shape[1]
    for col in range(n_state):
        big = active[:, col]
        if big.any():
            xi[big, col] = _lstsq_col(
                theta[:, big], dz[:, col : col + 1], ridge
            ).squeeze(-1)
    return xi


def _is_converged(X: torch.Tensor, X_prev: torch.Tensor, abstol: float, reltol: float) -> bool:
    delta = torch.norm(X - X_prev)
    if delta < abstol:
        return True
    if delta / (torch.norm(X) + 1e-12) < reltol:
        return True
    return False


# ---------------------------------------------------------------------------
# AdSR
# ---------------------------------------------------------------------------


class AdSR:
    """Adaptive Sparse Regression (AdSR).

    Iteratively thresholds a least-squares solution and re-fits the
    remaining terms, selecting the sparsity level via an information
    criterion evaluated on a held-out validation split.

    Parameters
    ----------
    ic : {'slic', 'aicc', 'bic'}
        Information criterion used for model selection.
    max_iter : int
        Number of outer threshold-sweep iterations.
    c : float
        Optional penalty that scales with the condition number of the
        training data (when ``use_cond=True``).
    train_pct : float
        Percentage of observations used for training; the rest form the
        validation set.
    abs_tol : float
        Absolute convergence tolerance on the training prediction.
    rel_tol : float
        Relative convergence tolerance on the training prediction.
    ridge : float
        Ridge (L2) penalty for the least-squares sub-problems.
    use_cond : bool
        If True, scale ``c`` by the condition number of the training
        library matrix.
    active_set : torch.Tensor or None
        Boolean mask (n_terms, n_states) of terms that are allowed to be
        nonzero.  If None, all terms are active.

    Attributes
    ----------
    coef_ : torch.Tensor of shape (n_terms, n_states)
        Fitted sparse coefficient matrix.
    score_ : float
        Best information-criterion score achieved.
    """

    def __init__(
        self,
        ic: Literal["slic", "aicc", "bic"] = "slic",
        max_iter: int = 10,
        c: float = 0.0,
        train_pct: float = 80.0,
        abs_tol: float = 1e-7,
        rel_tol: float = 1e-7,
        ridge: float = 0.0,
        use_cond: bool = True,
        active_set: torch.Tensor | None = None,
    ):
        if ic not in ("slic", "aicc", "bic"):
            raise ValueError(f"ic must be 'slic', 'aicc', or 'bic', got {ic!r}")
        self.ic = ic
        self.max_iter = max_iter
        self.c = c
        self.train_pct = train_pct
        self.abs_tol = abs_tol
        self.rel_tol = rel_tol
        self.ridge = ridge
        self.use_cond = use_cond
        self.active_set = active_set

    def fit(self, theta: torch.Tensor, y: torch.Tensor) -> "AdSR":
        """Fit the sparse regression model.

        Args:
            theta: Library matrix of shape (n_obs, n_terms).
            y: Target matrix of shape (n_obs, n_states).

        Returns:
            self
        """
        n_obs, n_state = y.shape
        bag_size = int(math.floor(self.train_pct * n_obs / 100))

        perm = torch.randperm(n_obs, device=y.device)
        train_idx = perm[:bag_size]
        test_idx = perm[bag_size:]

        theta_train, theta_test = theta[train_idx], theta[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        n_test = y_test.shape[0]

        cond = torch.linalg.cond(theta_train).item() if self.use_cond else 1.0
        eta = self.c * cond

        def _score(xi: torch.Tensor) -> float:
            k = int((xi.abs() > 0).sum().item()) + 1
            rss = torch.sum((y_test - theta_test @ xi) ** 2).item()
            adj = rss + eta
            if self.ic == "slic":
                return n_test * math.log(k * adj / n_test)
            elif self.ic == "aicc":
                denom = max(n_test - k - 1, 1)
                return n_test * math.log(adj / n_test) + 2 * k * n_test / denom
            else:  # bic
                return n_test * math.log(adj / n_test) + k * math.log(n_test)

        # Initial least-squares estimate
        xi = _lstsq_col(theta_train, y_train, self.ridge)
        active = (
            self.active_set.to(theta.device)
            if self.active_set is not None
            else torch.ones_like(xi, dtype=torch.bool)
        )
        xi = _update_active_coefs(theta_train, y_train, xi, active, self.ridge)

        min_score = float("inf")
        prev_small: torch.Tensor | None = None
        X_prev = theta_train @ xi

        for _ in range(self.max_iter):
            nonzero_vals = xi[xi != 0].abs()
            if nonzero_vals.numel() == 0:
                break

            lam_min = nonzero_vals.min().item()
            lam_max = nonzero_vals.max().item()
            ratio = lam_max / (lam_min + 1e-12)
            n_lambdas = max(2, abs(1000 * int(math.ceil(math.log10(ratio + 1e-10)))))
            lambdas = torch.linspace(lam_min, lam_max, n_lambdas, device=y.device)

            for lam in lambdas:
                temp_xi = xi.clone()
                small = temp_xi.abs() < lam.item()

                if prev_small is not None and torch.equal(small, prev_small):
                    continue

                temp_xi[small] = 0.0
                # All-zero column → skip this threshold
                if (temp_xi.abs() == 0).all(dim=0).any():
                    break

                temp_xi = _update_active_coefs(
                    theta_train, y_train, temp_xi, ~small, self.ridge
                )
                prev_small = small

                s = _score(temp_xi)
                if s < min_score:
                    xi = temp_xi.clone()
                    min_score = s

            X_curr = theta_train @ xi
            if _is_converged(X_curr, X_prev, self.abs_tol, self.rel_tol):
                break
            X_prev = X_curr

        self.coef_ = xi
        self.score_ = min_score
        return self


# ---------------------------------------------------------------------------
# EnAdSR
# ---------------------------------------------------------------------------


class EnAdSR:
    """Ensemble Adaptive Sparse Regression (EnAdSR).

    Runs :class:`AdSR` on multiple random train/validation splits and
    aggregates the results via inclusion probabilities.  Terms whose
    inclusion probability falls below ``tol`` are pruned; the remaining
    terms are re-fitted on all data.

    Parameters
    ----------
    ic : {'slic', 'aicc', 'bic'}
        Information criterion forwarded to each :class:`AdSR` run.
    num_batches : int
        Number of ensemble members (independent AdSR runs).
    tol : float
        Inclusion probability threshold; terms below this are zeroed.
    **kwargs
        Additional keyword arguments forwarded to :class:`AdSR`.

    Attributes
    ----------
    coef_ : torch.Tensor of shape (n_terms, n_states)
        Fitted sparse coefficient matrix.
    scores_ : torch.Tensor of shape (num_batches,)
        Per-batch information-criterion scores.
    inclusion_probs_ : torch.Tensor of shape (n_terms, n_states)
        Fraction of ensemble members in which each coefficient was nonzero.
    """

    def __init__(
        self,
        ic: Literal["slic", "aicc", "bic"] = "slic",
        num_batches: int = 10,
        tol: float = 0.7,
        **kwargs,
    ):
        self.ic = ic
        self.num_batches = num_batches
        self.tol = tol
        self.kwargs = kwargs

    def fit(self, theta: torch.Tensor, y: torch.Tensor) -> "EnAdSR":
        """Fit the ensemble sparse regression model.

        Args:
            theta: Library matrix of shape (n_obs, n_terms).
            y: Target matrix of shape (n_obs, n_states).

        Returns:
            self
        """
        n_terms = theta.shape[1]
        n_states = y.shape[1]

        xi_stack = torch.zeros(n_terms, n_states, self.num_batches, dtype=theta.dtype, device=theta.device)
        scores = torch.zeros(self.num_batches, dtype=theta.dtype, device=theta.device)

        for i in range(self.num_batches):
            reg = AdSR(ic=self.ic, **self.kwargs)
            reg.fit(theta, y)
            xi_stack[:, :, i] = reg.coef_
            scores[i] = reg.score_

        # Inclusion probabilities
        active = xi_stack.abs() > 0
        ips = active.float().mean(dim=2)  # (n_terms, n_states)

        # Ensemble mean over active entries
        n_active = active.float().sum(dim=2).clamp(min=1.0)
        xi_mean = xi_stack.sum(dim=2) / n_active
        xi_mean[ips < self.tol] = 0.0

        # Final re-fit on selected (active) terms using all data
        final_active = xi_mean.abs() > 0
        xi_final = _update_active_coefs(theta, y, xi_mean, final_active, ridge=self.kwargs.get("ridge", 0.0))

        self.coef_ = xi_final
        self.scores_ = scores
        self.inclusion_probs_ = ips
        return self


In [ ]:
%%writefile pydme/models.py
"""Core pydme model classes.

All models follow a scikit-learn-style API:
  - Hyperparameters are set in ``__init__``.
  - ``fit(X)`` trains the model and stores results as attributes ending in ``_``.
  - ``fit`` returns ``self`` for method chaining.

Three model classes are provided:

``TMI``
    Core Temporal Matrix Integration: decomposes X ≈ softmax(-Y @ z.T)
    using KL divergence.  Supports 2-D data (n_features, n_timepoints) and
    3-D data (n_features, n_timepoints, n_trials).  Optionally learns a
    per-feature bias term E.

``TMIWithDynamics``
    Extends ``TMI`` with a sparse SINDy-style dynamics constraint on z:
        DW @ z ≈ W @ poly_library(z, N) @ xi_z
    xi_z is discovered via ensemble sparse regression (EnAdSR).

``TMIWithFeatureModel``
    Extends ``TMIWithDynamics`` with an additional sparse feature model on Y:
        Y ≈ theta_y @ xi_y
    Both xi_z and xi_y are discovered via EnAdSR.
"""

from __future__ import annotations

import math
from typing import Sequence

import torch
import torch.nn as nn

from .library import poly_library
from .losses import kld_loss
from .sparse import EnAdSR, _update_active_coefs


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def _to_tensor(x, dtype=torch.float64, device=None) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        t = x.to(dtype)
    else:
        t = torch.tensor(x, dtype=dtype)
    return t if device is None else t.to(device)


def _svd_init_2d(X: torch.Tensor, K: int):
    """SVD initialisation for 2-D data (no bias term)."""
    U, _, Vh = torch.linalg.svd(X, full_matrices=False)
    Y = -U[:, :K]       # (a, K)
    z = -Vh.T[:, :K]    # (alpha, K)
    return z, Y


def _svd_init_2d_with_E(X: torch.Tensor, K: int):
    """SVD initialisation for 2-D data with a mean (bias) term."""
    Xavg = X.mean(dim=1, keepdim=True)          # (a, 1)
    U, _, Vh = torch.linalg.svd(X - Xavg, full_matrices=False)
    Y = -U[:, :K]
    z = -Vh.T[:, :K]
    return z, Y, Xavg


# ---------------------------------------------------------------------------
# TMI (core)
# ---------------------------------------------------------------------------


class TMI:
    """Temporal Matrix Integration (TMI).

    Decomposes a non-negative data matrix X as::

        q = softmax_col( -Y @ z.T )          # 2-D
        q[a,t,j] = softmax_col( -Y[a,:] @ z[t,:,j] )  # 3-D

    and minimises the column-normalised KL divergence D_KL(X || q).

    Parameters
    ----------
    K : int
        Latent dimensionality.
    normalize : bool
        Column-normalise X (and q) so that each column is a probability
        distribution.
    max_iters : int
        Maximum number of gradient-descent steps.
    tol : float
        Stop when the RMS gradient magnitude drops below this value.
    lr : float
        SGD learning rate.
    use_E : bool
        If True, learn a per-feature bias term E (adds mean-field offset).
    l2_z : float
        L2 regularisation weight on z.
    l2_y : float
        L2 regularisation weight on Y.
    use_rand_init : bool
        Random initialisation; otherwise use SVD-based initialisation.
    seed : int
        Random seed (used for both initialisation and sparse regression).
    track_latents : bool
        Record z and Y at every iteration (memory-intensive).
    verbose : bool
        Print progress every 1 000 iterations.

    Attributes
    ----------
    z_ : torch.Tensor
        Fitted time-domain latents.
    Y_ : torch.Tensor
        Fitted feature-domain latents.
    E_ : torch.Tensor or None
        Fitted bias (None if ``use_E=False``).
    q_ : torch.Tensor
        Model reconstruction of X.
    grads_ : torch.Tensor
        RMS gradient magnitude per iteration.
    losses_ : torch.Tensor
        Loss value per iteration.
    z_history_, Y_history_ : list
        Per-iteration snapshots (only if ``track_latents=True``).
    """

    def __init__(
        self,
        K: int,
        *,
        normalize: bool = True,
        max_iters: int = 100_000,
        tol: float = 1e-3,
        lr: float = 1e-3,
        use_E: bool = False,
        l2_z: float = 0.0,
        l2_y: float = 0.0,
        use_rand_init: bool = False,
        seed: int = 0,
        track_latents: bool = False,
        verbose: bool = True,
    ):
        self.K = K
        self.normalize = normalize
        self.max_iters = max_iters
        self.tol = tol
        self.lr = lr
        self.use_E = use_E
        self.l2_z = l2_z
        self.l2_y = l2_y
        self.use_rand_init = use_rand_init
        self.seed = seed
        self.track_latents = track_latents
        self.verbose = verbose

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _compute_q(
        self,
        Y: torch.Tensor,
        z: torch.Tensor,
        E: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Compute the model reconstruction q from current latents."""
        if z.ndim == 2:
            C = Y @ z.T                                    # (a, alpha)
            bias = E if E is not None else 0.0
            q = torch.exp(-C - bias)
        else:
            # z: (alpha, K, trials) — Julia convention keeps time dim first
            # but here z is (T, K, trials); Y is (a, K)
            C = torch.einsum("ak,tkj->atj", Y, z)          # (a, T, trials)
            bias = E[:, None, None] if E is not None else 0.0
            q = torch.exp(-C - bias)

        if self.normalize:
            q = q / q.sum(dim=0, keepdim=True)
        return q

    def _init_params(self, X: torch.Tensor):
        """Initialise z, Y (and optionally E) from X."""
        device, dtype = X.device, X.dtype
        a = X.shape[0]
        K = self.K
        is_3d = X.ndim == 3

        torch.manual_seed(self.seed)

        if self.use_rand_init:
            if is_3d:
                alpha, n_trials = X.shape[1], X.shape[2]
                z = torch.randn(alpha, K, n_trials, dtype=dtype, device=device)
                Y = torch.randn(a, K, dtype=dtype, device=device)
            else:
                alpha = X.shape[1]
                z = torch.randn(alpha, K, dtype=dtype, device=device)
                Y = torch.randn(a, K, dtype=dtype, device=device)
            E = torch.randn(a, 1, dtype=dtype, device=device) if self.use_E else None

        else:
            if is_3d:
                alpha, n_trials = X.shape[1], X.shape[2]
                Xavg = X.mean(dim=(1, 2), keepdim=False)   # (a,)
                z = torch.zeros(alpha, K, n_trials, dtype=dtype, device=device)
                Y_stack = torch.zeros(a, K, n_trials, dtype=dtype, device=device)
                dX = X - Xavg[:, None, None]
                for j in range(n_trials):
                    U, _, Vh = torch.linalg.svd(dX[:, :, j], full_matrices=False)
                    z[:, :, j] = -Vh.T[:, :K]
                    Y_stack[:, :, j] = -U[:, :K]
                Y = Y_stack.mean(dim=2)
                E = Xavg if self.use_E else None
            else:
                if self.use_E:
                    z, Y, E = _svd_init_2d_with_E(X, K)
                else:
                    z, Y = _svd_init_2d(X, K)
                    E = None

        return z, Y, E

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def fit(
        self,
        X,
        *,
        z_init=None,
        Y_init=None,
        E_init=None,
    ) -> "TMI":
        """Fit the TMI model to data.

        Parameters
        ----------
        X : array-like of shape (a, alpha) or (a, alpha, T)
            Input data matrix (or tensor).
        z_init : array-like, optional
            Initial value for z.
        Y_init : array-like, optional
            Initial value for Y.
        E_init : array-like, optional
            Initial value for E (only used when ``use_E=True``).

        Returns
        -------
        self
        """
        X = _to_tensor(X)
        device, dtype = X.device, X.dtype

        if self.normalize:
            X = X / (X.sum(dim=0, keepdim=True) + 1e-12)

        # Initialise latents
        if z_init is None or Y_init is None:
            z0, Y0, E0 = self._init_params(X)
        else:
            z0 = _to_tensor(z_init, dtype=dtype, device=device)
            Y0 = _to_tensor(Y_init, dtype=dtype, device=device)
            E0 = _to_tensor(E_init, dtype=dtype, device=device) if E_init is not None else None

        z = nn.Parameter(z0)
        Y = nn.Parameter(Y0)
        params = [z, Y]

        if self.use_E:
            E_init_val = E0 if E0 is not None else torch.zeros(X.shape[0], 1, dtype=dtype, device=device)
            E = nn.Parameter(E_init_val)
            params.append(E)
        else:
            E = None

        optimizer = torch.optim.SGD(params, lr=self.lr)

        grads_hist: list[float] = []
        losses_hist: list[float] = []
        z_hist = [] if self.track_latents else None
        Y_hist = [] if self.track_latents else None

        for i in range(1, self.max_iters + 1):
            if self.track_latents:
                z_hist.append(z.detach().clone())
                Y_hist.append(Y.detach().clone())

            optimizer.zero_grad()
            q = self._compute_q(Y, z, E)
            loss = kld_loss(X, q)
            if self.l2_z > 0:
                loss = loss + 0.5 * self.l2_z * z.pow(2).sum()
            if self.l2_y > 0:
                loss = loss + 0.5 * self.l2_y * Y.pow(2).sum()
            loss.backward()

            # RMS gradient magnitude as convergence signal
            mse_grad = math.sqrt(
                sum(p.grad.abs().mean().item() ** 2 for p in params if p.grad is not None)
            )
            grads_hist.append(mse_grad)
            losses_hist.append(loss.item())

            optimizer.step()

            if i % 1000 == 0 and self.verbose:
                print(f"iter => {i:>6d}  mse_grad => {mse_grad:.6e}  loss => {loss.item():.6f}")

            if mse_grad < self.tol:
                if self.verbose:
                    print(f"Converged at iter {i}  mse_grad => {mse_grad:.6e}")
                break

        self.z_ = z.detach()
        self.Y_ = Y.detach()
        self.E_ = E.detach() if E is not None else None
        with torch.no_grad():
            self.q_ = self._compute_q(self.Y_, self.z_, self.E_)
        self.grads_ = torch.tensor(grads_hist)
        self.losses_ = torch.tensor(losses_hist)
        if self.track_latents:
            self.z_history_ = z_hist
            self.Y_history_ = Y_hist

        return self

    def transform(
        self,
        z=None,
        Y=None,
        E=None,
    ) -> torch.Tensor:
        """Compute the model reconstruction from latents.

        Uses the fitted latents by default; pass explicit values to override.
        """
        z = self.z_ if z is None else _to_tensor(z)
        Y = self.Y_ if Y is None else _to_tensor(Y)
        E = self.E_ if E is None else _to_tensor(E)
        with torch.no_grad():
            return self._compute_q(Y, z, E)


# ---------------------------------------------------------------------------
# TMIWithDynamics
# ---------------------------------------------------------------------------


class TMIWithDynamics(TMI):
    """TMI with a sparse SINDy-like dynamics constraint on z.

    In addition to the KLD reconstruction loss, the model penalises
    violations of::

        DW @ z ≈ W @ poly_library(z, N) @ xi_z

    where ``xi_z`` is a sparse coefficient matrix discovered via
    :class:`~pydme.sparse.EnAdSR`.

    Parameters
    ----------
    K : int
        Latent dimensionality.
    W : array-like of shape (T, T)
        Galerkin weight matrix applied to the library.
    DW : array-like of shape (T, T)
        Galerkin weight matrix applied to the derivative of z.
    N : array-like of shape (L, K)
        Polynomial exponent matrix defining the library terms.
    lambda_z : float
        Weight of the dynamics constraint loss.
    l2_smooth : float
        Weight of a smoothness penalty ``||DW @ z||^2``.
    sparse_iter : int
        Run :class:`~pydme.sparse.EnAdSR` every this many iterations to
        re-discover the active support of xi_z; between sparse regression
        steps, the active coefficients are updated by least-squares.
    inclusion_tol : float
        EnAdSR inclusion-probability threshold.
    ridge : float
        Ridge penalty for the least-squares sub-problems inside EnAdSR.
    c_z : float
        EnAdSR condition-number scaling parameter for xi_z.
    **kwargs
        Additional keyword arguments forwarded to :class:`TMI`.

    Attributes
    ----------
    xi_z_ : torch.Tensor of shape (L, K)
        Fitted sparse dynamics model coefficients.
    xi_z_history_ : list
        Per-iteration snapshots of xi_z (only if ``track_latents=True``).
    """

    def __init__(
        self,
        K: int,
        W,
        DW,
        N,
        *,
        lambda_z: float = 10.0,
        l2_smooth: float = 0.0,
        sparse_iter: int = 10_000,
        inclusion_tol: float = 0.7,
        ridge: float = 0.0,
        c_z: float = 1e-4,
        **kwargs,
    ):
        super().__init__(K, **kwargs)
        self.W = W
        self.DW = DW
        self.N = N
        self.lambda_z = lambda_z
        self.l2_smooth = l2_smooth
        self.sparse_iter = sparse_iter
        self.inclusion_tol = inclusion_tol
        self.ridge = ridge
        self.c_z = c_z

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _dynamics_loss(
        self,
        z: torch.Tensor,
        xi_z: torch.Tensor,
        W: torch.Tensor,
        DW: torch.Tensor,
        N: torch.Tensor,
    ) -> torch.Tensor:
        """Compute the dynamics constraint loss."""
        if z.ndim == 2:
            theta = poly_library(z, N)          # (T, L)
            dzdt = DW @ z                        # (T, K)
            theta_w = W @ theta                  # (T, L)
            residual = dzdt - theta_w @ xi_z     # (T, K)
            loss = 0.5 * self.lambda_z * residual.pow(2).sum()
            if self.l2_smooth > 0:
                loss = loss + 0.5 * self.l2_smooth * dzdt.pow(2).sum()
        else:
            # z: (T, K, trials)
            theta = poly_library(z, N)                                     # (T, L, trials)
            dzdt = torch.einsum("tl,lkj->tkj", DW, z)                     # (T, K, trials)
            theta_w = torch.einsum("tl,lkj->tkj", W, theta)               # (T, L, trials)
            pred = torch.einsum("tlj,lk->tkj", theta_w, xi_z)             # (T, K, trials)
            residual = dzdt - pred
            loss = 0.5 * self.lambda_z * residual.pow(2).sum()
            if self.l2_smooth > 0:
                loss = loss + 0.5 * self.l2_smooth * dzdt.pow(2).sum()
        return loss

    @torch.no_grad()
    def _get_theta_w_dzdt(
        self,
        z: torch.Tensor,
        W: torch.Tensor,
        DW: torch.Tensor,
        N: torch.Tensor,
    ):
        """Return the (possibly reshaped) library and derivative arrays
        suitable for passing to sparse regression."""
        if z.ndim == 2:
            theta = poly_library(z, N)
            return W @ theta, DW @ z           # (T, L), (T, K)
        else:
            theta = poly_library(z, N)                          # (T, L, trials)
            theta_w = torch.einsum("tl,lkj->tkj", W, theta)    # (T, L, trials)
            dzdt = torch.einsum("tl,lkj->tkj", DW, z)          # (T, K, trials)
            T, L, trials = theta_w.shape
            K = dzdt.shape[1]
            theta_w_2d = theta_w.permute(0, 2, 1).reshape(T * trials, L)
            dzdt_2d = dzdt.permute(0, 2, 1).reshape(T * trials, K)
            return theta_w_2d, dzdt_2d

    def _run_enadsr_z(self, theta_w, dzdt, active):
        return EnAdSR(
            ic="slic",
            num_batches=10,
            tol=self.inclusion_tol,
            max_iter=5,
            c=self.c_z,
            ridge=self.ridge,
            active_set=active,
        ).fit(theta_w, dzdt).coef_

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def fit(
        self,
        X,
        *,
        z_init=None,
        Y_init=None,
        E_init=None,
        xi_z_init=None,
    ) -> "TMIWithDynamics":
        """Fit the model.

        Parameters
        ----------
        X : array-like of shape (a, alpha) or (a, alpha, T)
        z_init, Y_init, E_init : optional initialisations.
        xi_z_init : array-like of shape (L, K), optional
            Initial sparse dynamics coefficients.

        Returns
        -------
        self
        """
        X = _to_tensor(X)
        device, dtype = X.device, X.dtype

        W = _to_tensor(self.W, dtype=dtype, device=device)
        DW = _to_tensor(self.DW, dtype=dtype, device=device)
        N = _to_tensor(self.N, dtype=dtype, device=device)

        if self.normalize:
            X = X / (X.sum(dim=0, keepdim=True) + 1e-12)

        if z_init is None or Y_init is None:
            z0, Y0, E0 = self._init_params(X)
        else:
            z0 = _to_tensor(z_init, dtype=dtype, device=device)
            Y0 = _to_tensor(Y_init, dtype=dtype, device=device)
            E0 = _to_tensor(E_init, dtype=dtype, device=device) if E_init is not None else None

        z = nn.Parameter(z0)
        Y = nn.Parameter(Y0)
        params = [z, Y]
        if self.use_E:
            E_val = E0 if E0 is not None else torch.zeros(X.shape[0], 1, dtype=dtype, device=device)
            E = nn.Parameter(E_val)
            params.append(E)
        else:
            E = None

        # Initialise xi_z
        with torch.no_grad():
            theta_w, dzdt = self._get_theta_w_dzdt(z.data, W, DW, N)
            if xi_z_init is not None:
                xi_z = _to_tensor(xi_z_init, dtype=dtype, device=device)
            else:
                xi_z = torch.linalg.lstsq(theta_w, dzdt).solution
            xi_z_active = xi_z.abs() > 0

        optimizer = torch.optim.SGD(params, lr=self.lr)

        grads_hist: list[float] = []
        losses_hist: list[float] = []
        z_hist = [] if self.track_latents else None
        Y_hist = [] if self.track_latents else None
        xi_z_hist = [] if self.track_latents else None

        for i in range(1, self.max_iters + 1):
            if self.track_latents:
                z_hist.append(z.detach().clone())
                Y_hist.append(Y.detach().clone())
                xi_z_hist.append(xi_z.clone())

            optimizer.zero_grad()
            q = self._compute_q(Y, z, E)
            loss = kld_loss(X, q)
            if self.lambda_z > 0:
                loss = loss + self._dynamics_loss(z, xi_z, W, DW, N)
            if self.l2_z > 0:
                loss = loss + 0.5 * self.l2_z * z.pow(2).sum()
            if self.l2_y > 0:
                loss = loss + 0.5 * self.l2_y * Y.pow(2).sum()
            loss.backward()

            mse_grad = math.sqrt(
                sum(p.grad.abs().mean().item() ** 2 for p in params if p.grad is not None)
            )
            grads_hist.append(mse_grad)
            losses_hist.append(loss.item())

            optimizer.step()

            # Update xi_z
            with torch.no_grad():
                theta_w, dzdt = self._get_theta_w_dzdt(z.data, W, DW, N)
                if self.lambda_z > 0 and i % self.sparse_iter == 0:
                    xi_z = self._run_enadsr_z(theta_w, dzdt, xi_z_active)
                    xi_z_active = xi_z.abs() > 0
                else:
                    xi_z = _update_active_coefs(theta_w, dzdt, xi_z, xi_z_active, self.ridge)

            if i % 1000 == 0 and self.verbose:
                print(f"iter => {i:>6d}  mse_grad => {mse_grad:.6e}  loss => {loss.item():.6f}")

            if mse_grad < self.tol:
                if self.verbose:
                    print(f"Converged at iter {i}  mse_grad => {mse_grad:.6e}")
                break

        self.z_ = z.detach()
        self.Y_ = Y.detach()
        self.E_ = E.detach() if E is not None else None
        with torch.no_grad():
            self.q_ = self._compute_q(self.Y_, self.z_, self.E_)
        self.xi_z_ = xi_z
        self.grads_ = torch.tensor(grads_hist)
        self.losses_ = torch.tensor(losses_hist)
        if self.track_latents:
            self.z_history_ = z_hist
            self.Y_history_ = Y_hist
            self.xi_z_history_ = xi_z_hist

        return self


# ---------------------------------------------------------------------------
# TMIWithFeatureModel
# ---------------------------------------------------------------------------


class TMIWithFeatureModel(TMIWithDynamics):
    """TMI with sparse dynamics on z and a sparse feature model on Y.

    Extends :class:`TMIWithDynamics` by penalising deviations of Y from a
    sparse linear model over a given feature library ``theta_y``::

        Y ≈ theta_y @ xi_y

    Both ``xi_z`` and ``xi_y`` are discovered via
    :class:`~pydme.sparse.EnAdSR`.

    Parameters
    ----------
    K : int
        Latent dimensionality.
    W, DW, N : array-like
        Galerkin matrices and polynomial exponent matrix (see
        :class:`TMIWithDynamics`).
    theta_y : array-like of shape (alpha, P)
        Feature library matrix (not learned; provided by the user).
    lambda_y : float
        Weight of the feature model loss ``||Y - theta_y @ xi_y||^2``.
    c_y : float
        EnAdSR condition-number scaling for xi_y.
    **kwargs
        Additional keyword arguments forwarded to :class:`TMIWithDynamics`.

    Attributes
    ----------
    xi_y_ : torch.Tensor of shape (P, K)
        Fitted sparse feature model coefficients.
    xi_y_history_ : list
        Per-iteration snapshots of xi_y (only if ``track_latents=True``).
    """

    def __init__(
        self,
        K: int,
        W,
        DW,
        N,
        theta_y,
        *,
        lambda_y: float = 10.0,
        c_y: float = 1e-4,
        **kwargs,
    ):
        super().__init__(K, W, DW, N, **kwargs)
        self.theta_y = theta_y
        self.lambda_y = lambda_y
        self.c_y = c_y

    def _feature_loss(
        self,
        Y: torch.Tensor,
        xi_y: torch.Tensor,
        theta_y: torch.Tensor,
    ) -> torch.Tensor:
        residual = Y - theta_y @ xi_y
        return 0.5 * self.lambda_y * residual.pow(2).sum()

    def _run_enadsr_y(self, theta_y, Y, active):
        return EnAdSR(
            ic="slic",
            num_batches=10,
            tol=self.inclusion_tol,
            max_iter=5,
            c=self.c_y,
            ridge=self.ridge,
            active_set=active,
        ).fit(theta_y, Y).coef_

    def fit(
        self,
        X,
        *,
        z_init=None,
        Y_init=None,
        E_init=None,
        xi_z_init=None,
        xi_y_init=None,
    ) -> "TMIWithFeatureModel":
        """Fit the model.

        Parameters
        ----------
        X : array-like of shape (a, alpha) or (a, alpha, T)
        z_init, Y_init, E_init : optional initialisations.
        xi_z_init : array-like of shape (L, K), optional
        xi_y_init : array-like of shape (P, K), optional

        Returns
        -------
        self
        """
        X = _to_tensor(X)
        device, dtype = X.device, X.dtype

        W = _to_tensor(self.W, dtype=dtype, device=device)
        DW = _to_tensor(self.DW, dtype=dtype, device=device)
        N = _to_tensor(self.N, dtype=dtype, device=device)
        theta_y = _to_tensor(self.theta_y, dtype=dtype, device=device)

        if self.normalize:
            X = X / (X.sum(dim=0, keepdim=True) + 1e-12)

        if z_init is None or Y_init is None:
            z0, Y0, E0 = self._init_params(X)
        else:
            z0 = _to_tensor(z_init, dtype=dtype, device=device)
            Y0 = _to_tensor(Y_init, dtype=dtype, device=device)
            E0 = _to_tensor(E_init, dtype=dtype, device=device) if E_init is not None else None

        z = nn.Parameter(z0)
        Y = nn.Parameter(Y0)
        params = [z, Y]
        if self.use_E:
            E_val = E0 if E0 is not None else torch.zeros(X.shape[0], 1, dtype=dtype, device=device)
            E = nn.Parameter(E_val)
            params.append(E)
        else:
            E = None

        # Initialise xi_z and xi_y
        with torch.no_grad():
            theta_w, dzdt = self._get_theta_w_dzdt(z.data, W, DW, N)
            xi_z = (
                _to_tensor(xi_z_init, dtype=dtype, device=device)
                if xi_z_init is not None
                else torch.linalg.lstsq(theta_w, dzdt).solution
            )
            xi_z_active = xi_z.abs() > 0

            xi_y = (
                _to_tensor(xi_y_init, dtype=dtype, device=device)
                if xi_y_init is not None
                else torch.linalg.lstsq(theta_y, Y.data).solution
            )
            xi_y_active = xi_y.abs() > 0

        optimizer = torch.optim.SGD(params, lr=self.lr)

        grads_hist: list[float] = []
        losses_hist: list[float] = []
        z_hist = [] if self.track_latents else None
        Y_hist = [] if self.track_latents else None
        xi_z_hist = [] if self.track_latents else None
        xi_y_hist = [] if self.track_latents else None

        for i in range(1, self.max_iters + 1):
            if self.track_latents:
                z_hist.append(z.detach().clone())
                Y_hist.append(Y.detach().clone())
                xi_z_hist.append(xi_z.clone())
                xi_y_hist.append(xi_y.clone())

            optimizer.zero_grad()
            q = self._compute_q(Y, z, E)
            loss = kld_loss(X, q)
            if self.lambda_z > 0:
                loss = loss + self._dynamics_loss(z, xi_z, W, DW, N)
            if self.lambda_y > 0:
                loss = loss + self._feature_loss(Y, xi_y, theta_y)
            if self.l2_z > 0:
                loss = loss + 0.5 * self.l2_z * z.pow(2).sum()
            if self.l2_y > 0:
                loss = loss + 0.5 * self.l2_y * Y.pow(2).sum()
            loss.backward()

            mse_grad = math.sqrt(
                sum(p.grad.abs().mean().item() ** 2 for p in params if p.grad is not None)
            )
            grads_hist.append(mse_grad)
            losses_hist.append(loss.item())

            optimizer.step()

            # Update xi_z and xi_y
            with torch.no_grad():
                theta_w, dzdt = self._get_theta_w_dzdt(z.data, W, DW, N)
                if self.lambda_z > 0 and i % self.sparse_iter == 0:
                    xi_z = self._run_enadsr_z(theta_w, dzdt, xi_z_active)
                    xi_z_active = xi_z.abs() > 0
                    xi_y = self._run_enadsr_y(theta_y, Y.detach(), xi_y_active)
                    xi_y_active = xi_y.abs() > 0
                else:
                    xi_z = _update_active_coefs(theta_w, dzdt, xi_z, xi_z_active, self.ridge)
                    xi_y = _update_active_coefs(theta_y, Y.detach(), xi_y, xi_y_active, self.ridge)

            if i % 1000 == 0 and self.verbose:
                print(f"iter => {i:>6d}  mse_grad => {mse_grad:.6e}  loss => {loss.item():.6f}")

            if mse_grad < self.tol:
                if self.verbose:
                    print(f"Converged at iter {i}  mse_grad => {mse_grad:.6e}")
                break

        self.z_ = z.detach()
        self.Y_ = Y.detach()
        self.E_ = E.detach() if E is not None else None
        with torch.no_grad():
            self.q_ = self._compute_q(self.Y_, self.z_, self.E_)
        self.xi_z_ = xi_z
        self.xi_y_ = xi_y
        self.grads_ = torch.tensor(grads_hist)
        self.losses_ = torch.tensor(losses_hist)
        if self.track_latents:
            self.z_history_ = z_hist
            self.Y_history_ = Y_hist
            self.xi_z_history_ = xi_z_hist
            self.xi_y_history_ = xi_y_hist

        return self


# ---------------------------------------------------------------------------
# Hyperparameter utilities
# ---------------------------------------------------------------------------


def scan_hyperparams(
    X,
    model_cls,
    param_grid: dict[str, Sequence],
    **model_kwargs,
) -> list[dict]:
    """Grid search over hyperparameters.

    Parameters
    ----------
    X : array-like
        Input data.
    model_cls : type
        One of :class:`TMI`, :class:`TMIWithDynamics`,
        :class:`TMIWithFeatureModel`.
    param_grid : dict
        Mapping from constructor parameter names to lists of values to try.
        All combinations are evaluated.
    **model_kwargs
        Fixed keyword arguments forwarded to ``model_cls``.

    Returns
    -------
    list of dict
        Each entry contains ``'model'`` (fitted) and one key per grid
        parameter with the value used.

    Example
    -------
    >>> results = scan_hyperparams(
    ...     X, TMIWithFeatureModel,
    ...     param_grid={"lambda_z": [1., 10.], "lambda_y": [1., 10.]},
    ...     K=3, W=W, DW=DW, N=N, theta_y=theta_y,
    ... )
    """
    import itertools

    keys = list(param_grid.keys())
    values = list(param_grid.values())
    results = []

    for combo in itertools.product(*values):
        kw = dict(zip(keys, combo))
        print("  ".join(f"{k} => {v}" for k, v in kw.items()))
        m = model_cls(**kw, **model_kwargs)
        m.fit(X)
        results.append({"model": m, **kw})

    return results


def score_models(
    results: list[dict],
    N,
    W,
    DW,
    theta_y=None,
) -> list[float]:
    """Score a list of fitted models using the SLIC information criterion.

    Parameters
    ----------
    results : list of dict
        Output from :func:`scan_hyperparams`.
    N : array-like
        Polynomial exponent matrix (same as used during fitting).
    W, DW : array-like
        Galerkin matrices (same as used during fitting).
    theta_y : array-like or None
        Feature library (required when models have ``xi_y_``).

    Returns
    -------
    list of float
        SLIC score per model (lower is better).
    """
    scores = []
    for r in results:
        model = r["model"]
        z = model.z_
        dtype, device = z.dtype, z.device
        xi_z = model.xi_z_

        N_t = _to_tensor(N, dtype=dtype, device=device)
        W_t = _to_tensor(W, dtype=dtype, device=device)
        DW_t = _to_tensor(DW, dtype=dtype, device=device)

        if z.ndim == 2:
            theta = poly_library(z, N_t)
            theta_w = W_t @ theta
            dzdt = DW_t @ z
            n = z.shape[0]
        else:
            theta = poly_library(z, N_t)
            theta_w = torch.einsum("tl,lkj->tkj", W_t, theta)
            dzdt = torch.einsum("tl,lkj->tkj", DW_t, z)
            T, L, trials = theta_w.shape
            K = dzdt.shape[1]
            theta_w = theta_w.permute(0, 2, 1).reshape(T * trials, L)
            dzdt = dzdt.permute(0, 2, 1).reshape(T * trials, K)
            n = T * trials

        k_z = int((xi_z.abs() > 0).sum().item()) + 1
        rss_z = (dzdt - theta_w @ xi_z).pow(2).sum().item()
        score_total = n * math.log(max(k_z * rss_z / n, 1e-12))

        if theta_y is not None and hasattr(model, "xi_y_"):
            xi_y = model.xi_y_
            Y = model.Y_
            theta_y_t = _to_tensor(theta_y, dtype=dtype, device=device)
            n_y = Y.shape[0]
            k_y = int((xi_y.abs() > 0).sum().item()) + 1
            rss_y = (Y - theta_y_t @ xi_y).pow(2).sum().item()
            score_total += n_y * math.log(max(k_y * rss_y / n_y, 1e-12))

        scores.append(score_total)

    return scores


In [ ]:
%%writefile pydme/galerkin.py
"""Galerkin projection weight matrices for derivative-free dynamics estimation.

Rather than computing dz/dt by finite differences (which amplifies noise),
the Galerkin approach uses integration by parts on smooth local test functions
that vanish at the endpoints of each window:

    ∫ (dz/dt) w_i dt  =  -∫ z (dw_i/dt) dt

This yields the matrix equation:

    DW @ z  ≈  W @ theta(z) @ xi_z

where

  - W  (shape: n_windows × n_time) integrates z against each bump w_i
  - DW (shape: n_windows × n_time) integrates z against -dw_i/dt

Estimating xi_z via this system avoids differentiating z entirely, giving
a noise-robust identification of the latent dynamics.

Python port of ``src/build_weight_mats.jl``.
"""

from __future__ import annotations

from itertools import product as _iproduct

import numpy as np


# ---------------------------------------------------------------------------
# Weight functions
# ---------------------------------------------------------------------------


def _w(t: np.ndarray, p: int, a: float = -1.0, b: float = 1.0) -> np.ndarray:
    """Smooth bump function supported on (a, b), vanishing at both ends.

    w(t; p, a, b) = (2/(b-a))^(2p) * ((t-a)(b-t))^p
    """
    c = (2.0 / (b - a)) ** (2 * p)
    return c * ((t - a) * (b - t)) ** p


def _dw(t: np.ndarray, p: int, a: float = -1.0, b: float = 1.0) -> np.ndarray:
    """First derivative of the bump function with respect to t."""
    c = (2.0 / (b - a)) ** (2 * p)
    inner = (t - a) * (b - t)
    # d/dt [(t-a)(b-t)]^p = p * [(t-a)(b-t)]^(p-1) * (a+b-2t)
    return c * p * inner ** (p - 1) * (a + b - 2.0 * t)


def _d2w(t: np.ndarray, p: int, a: float = -1.0, b: float = 1.0) -> np.ndarray:
    """Second derivative of the bump function with respect to t."""
    c = (2.0 / (b - a)) ** (2 * p)
    inner = (t - a) * (b - t)
    d_inner = a + b - 2.0 * t  # d/dt [(t-a)(b-t)]
    # d/dt [p * inner^(p-1) * d_inner]
    return c * p * ((p - 1) * inner ** (p - 2) * d_inner ** 2 - 2.0 * inner ** (p - 1))


# ---------------------------------------------------------------------------
# Matrix builders
# ---------------------------------------------------------------------------


def build_weight_matrix(
    ts: np.ndarray,
    window: int,
    d: int,
    p: int = 10,
) -> np.ndarray:
    """Build a Galerkin projection weight matrix from a 1-D time grid.

    Each row of the output corresponds to one overlapping local window
    ``ts[i : i+window+1]``.  The bump function ``w`` (or its derivative)
    is evaluated on the scaled window coordinates and assembled into a full
    (n_windows × n_time) dense matrix multiplied by the trapezoidal weight
    ``dt / 2``.

    Parameters
    ----------
    ts : ndarray of shape (n,)
        Uniformly spaced time points.
    window : int
        Number of *gaps* in each local window, so each window spans
        ``window + 1`` consecutive time points.
    d : {0, 1, 2}
        Derivative order:

        - 0 → W  (smooth integration matrix; corresponds to ∫ z w dt)
        - 1 → DW (derivative matrix; corresponds to -∫ z dw/dt dt)
        - 2 → DDW (second-derivative matrix)

    p : int
        Smoothness parameter of the bump function.  Higher values
        concentrate weight toward the window centre.

    Returns
    -------
    ndarray of shape (n - window, n)
        Dense Galerkin weight matrix.

    Notes
    -----
    The convention matches ``BuildWeightMat`` in ``src/build_weight_mats.jl``:
    the matrix is returned pre-multiplied by ``dt / 2`` so that a simple
    matrix–vector product gives the discretised integral.
    """
    if d not in (0, 1, 2):
        raise ValueError(f"d must be 0, 1, or 2; got {d}")

    n = len(ts)
    dt = float(ts[1] - ts[0])
    n_rows = n - window
    W = np.zeros((n_rows, n))

    for i in range(n_rows):
        ts_i = ts[i : i + window + 1]
        a_i, b_i = float(ts_i[0]), float(ts_i[-1])
        span = b_i - a_i

        # Map ts_i linearly to [-1, 1]
        sc = ts_i * (2.0 / span) - (b_i + a_i) / span

        if d == 0:
            # W row: 2 * w(sc) / (2/span) = span * w(sc)
            W[i, i : i + window + 1] = 2.0 * _w(sc, p) / (2.0 / span)
        elif d == 1:
            # DW row: -2 * dw(sc)   (chain-rule factor from the scaling cancels)
            W[i, i : i + window + 1] = -2.0 * _dw(sc, p)
        else:  # d == 2
            # DDW row: -2 * d2w(sc) * (2/span)
            W[i, i : i + window + 1] = -2.0 * _d2w(sc, p) * (2.0 / span)

    return W * (dt / 2.0)


def build_disc_weight_matrix(ts: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Build simple discrete (finite-difference) weight matrices.

    A lightweight alternative to :func:`build_weight_matrix` for clean,
    noise-free data.  Uses forward differences to approximate dz/dt.

    Parameters
    ----------
    ts : ndarray of shape (n,)
        Time points (only length is used; values are ignored).

    Returns
    -------
    DW : ndarray of shape (n, n)
        Forward-difference matrix: ``(DW @ z)[i] = z[i+1] - z[i]``.
    W : ndarray of shape (n, n)
        Identity matrix (no smoothing).

    Notes
    -----
    Equivalent to ``BuildDiscWeightMat`` in ``src/build_weight_mats.jl``.
    The last row of DW is all zeros (no forward neighbour).
    """
    n = len(ts)
    DW = np.zeros((n, n))
    for i in range(n - 1):
        DW[i, i] = -1.0
        DW[i, i + 1] = 1.0
    W = np.eye(n)
    return DW, W


def build_poly_degree_matrix(K: int, order: int) -> np.ndarray:
    """Build the polynomial exponent matrix for a K-dimensional library.

    Returns an ``(L, K)`` integer array where each row is a multi-index
    ``(e_1, ..., e_K)`` with ``sum(e_k) <= order``.  Library term ``l``
    corresponds to ``z[:,0]**e[l,0] * z[:,1]**e[l,1] * ...``.

    Parameters
    ----------
    K : int
        Number of latent dimensions (columns of the exponent matrix).
    order : int
        Maximum total polynomial degree.  For example, with K=2 and
        order=2 the terms are: 1, z0, z1, z0², z0·z1, z1².

    Returns
    -------
    ndarray of shape (L, K), dtype float64
        Exponent matrix.  The number of terms is
        ``L = binomial(K + order, order)``.

    Examples
    --------
    >>> build_poly_degree_matrix(K=2, order=2)
    array([[0., 0.],
           [1., 0.],
           [0., 1.],
           [2., 0.],
           [1., 1.],
           [0., 2.]])
    """
    if order < 0:
        raise ValueError("order must be >= 0")
    terms = [
        combo
        for combo in _iproduct(range(order + 1), repeat=K)
        if sum(combo) <= order
    ]
    return np.array(terms, dtype=np.float64)


In [ ]:
%%writefile pydme/__init__.py
"""
pydme — Python Dynamic Mode Extraction

A PyTorch implementation of Temporal Matrix Integration (TMI) with
sparse dynamics and feature models.
"""

from .models import TMI, TMIWithDynamics, TMIWithFeatureModel
from .sparse import AdSR, EnAdSR
from .library import poly_library
from .losses import kld_loss
from .galerkin import build_weight_matrix, build_disc_weight_matrix, build_poly_degree_matrix

__all__ = [
    "TMI",
    "TMIWithDynamics",
    "TMIWithFeatureModel",
    "AdSR",
    "EnAdSR",
    "poly_library",
    "kld_loss",
    "build_weight_matrix",
    "build_disc_weight_matrix",
    "build_poly_degree_matrix",
]


In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pydme import (
    TMI, TMIWithDynamics,
    build_weight_matrix,
    build_disc_weight_matrix,
    build_poly_degree_matrix,
)
print("pydme loaded ✓  torch:", torch.__version__)

## 1 · Generate 1+1-D Gaussian diffusion data

Analytical solution of the diffusion equation with a point-source initial
condition:

$$u(x, t) = \frac{1}{\sqrt{4\pi D t}}\exp\!\left(-\frac{x^2}{4Dt}\right)$$

The data matrix **X** has shape `(n_spatial, n_time)`.  Each column is the
spatial probability distribution at one time step.

With **K = 1** latent dimension, the TMI model represents:

$$q(x, t) = \text{softmax}\bigl(-Y(x)\cdot z(t)\bigr)$$

The spatial mode $Y(x) \approx x^2$ and the scalar latent $z(t) \approx
1/(4Dt)$ capture the decaying precision of the Gaussian.  The latent ODE is
then $\dot z = -z^2 \cdot (1/z t) = -z/t$, recoverable by a degree-2
polynomial library.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
N_SPATIAL  = 60      # spatial grid points  (= "a" in the paper)
N_TIME     = 150     # time steps           (= "alpha")
D          = 0.5     # diffusion coefficient
X_RANGE    = (-6., 6.)
T_RANGE    = (0.2, 3.0)
NOISE_STD  = 0.005   # additive noise on X
SEED       = 42

rng  = np.random.default_rng(SEED)
x    = np.linspace(*X_RANGE, N_SPATIAL)
t    = np.linspace(*T_RANGE, N_TIME)

X = np.zeros((N_SPATIAL, N_TIME))
for j, tj in enumerate(t):
    var = 2.0 * D * tj                          # σ²(t) = 2Dt
    X[:, j] = np.exp(-x**2 / (2*var)) / np.sqrt(2*np.pi*var)

if NOISE_STD > 0:
    X = np.clip(X + NOISE_STD * rng.standard_normal(X.shape), 1e-10, None)

# Column-normalise → each snapshot is a probability distribution
X = X / X.sum(axis=0, keepdims=True)

print(f"X shape: {X.shape}  (n_spatial × n_time)")
print(f"D = {D},  t ∈ [{t[0]:.2f}, {t[-1]:.2f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

ax = axes[0]
im = ax.imshow(X, aspect="auto", origin="lower",
               extent=[t[0], t[-1], x[0], x[-1]], cmap="hot")
plt.colorbar(im, ax=ax, label="u(x,t)")
ax.set_title("Data  X  (n_spatial × n_time)", fontsize=11)
ax.set_xlabel("time t"); ax.set_ylabel("space x")

ax = axes[1]
for idx in [0, N_TIME//4, N_TIME//2, 3*N_TIME//4, -1]:
    ax.plot(x, X[:, idx], label=f"t={t[idx]:.2f}")
ax.set_title("Column snapshots of X", fontsize=11)
ax.set_xlabel("space x"); ax.set_ylabel("u(x,t)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2 · TMI (core) — no dynamics

Fit the basic TMI model to discover spatial mode **Y** and temporal
trajectory **z** purely from reconstruction quality.

In [ ]:
K = 1   # one latent dimension is enough for a 1-D Gaussian family

model_basic = TMI(
    K       = K,
    max_iters = 8_000,
    lr      = 5e-4,
    tol     = 1e-5,
    normalize = True,
    verbose = True,
)
model_basic.fit(X)

In [ ]:
q_basic = model_basic.q_.numpy()
Y_basic = model_basic.Y_.numpy().squeeze()   # (n_spatial,)
z_basic = model_basic.z_.numpy().squeeze()   # (n_time,)

# ── KLD ──────────────────────────────────────────────────────────────────
eps = 1e-12
kld = float(np.mean(np.sum(X * np.log((X + eps) / (q_basic + eps)), axis=0)))
print(f"KLD (mean per column, nats): {kld:.6f}")

# ── Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ax = axes[0]
im = ax.imshow(q_basic, aspect="auto", origin="lower",
               extent=[t[0], t[-1], x[0], x[-1]], cmap="hot")
plt.colorbar(im, ax=ax)
ax.set_title("TMI reconstruction  q", fontsize=11)
ax.set_xlabel("time t"); ax.set_ylabel("space x")

ax = axes[1]
ax.plot(x, Y_basic)
ax.set_title("Spatial mode  Y(x)  (≈ x²)", fontsize=11)
ax.set_xlabel("x"); ax.set_ylabel("Y")

ax = axes[2]
ax.plot(t, z_basic, label="TMI z")
true_z = 1.0 / (4.0 * D * t)
scale  = np.abs(z_basic).mean() / np.abs(true_z).mean()
ax.plot(t, scale * true_z, "--", label="1/(4Dt) scaled", lw=1.5)
ax.set_title("Temporal latent  z(t)  (≈ precision)", fontsize=11)
ax.set_xlabel("time t"); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3 · Galerkin weight matrices

Rather than computing $\dot z$ by finite differences (noise-amplifying),
the Galerkin projection uses **integration by parts** on smooth local bump
functions $w_i(t)$ that vanish at window endpoints:

$$\int \dot z\, w_i\, dt = -\int z\, \dot w_i\, dt$$

This gives the matrix system

$$\underbrace{\mathbf{DW}}_{\text{d=1}}\, z
  \;\approx\;
  \underbrace{\mathbf{W}}_{\text{d=0}}\, \boldsymbol{\theta}(z)\,\boldsymbol{\xi}_z$$

where **no derivative of z is ever taken explicitly**.

In [ ]:
WINDOW = 20   # number of time-step gaps per local window
P      = 10   # bump smoothness parameter

W  = build_weight_matrix(t, window=WINDOW, d=0, p=P)   # (n_time-WINDOW, n_time)
DW = build_weight_matrix(t, window=WINDOW, d=1, p=P)   # same shape
N  = build_poly_degree_matrix(K=K, order=2)             # {1, z, z²}

print(f"W  shape: {W.shape}")
print(f"DW shape: {DW.shape}")
print(f"N  (polynomial library exponents):\n{N}")

# Visualise one row of W and DW
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
row = W.shape[0] // 2
axes[0].plot(W[row]);  axes[0].set_title(f"W row {row}  (bump, d=0)");  axes[0].set_xlabel("time index")
axes[1].plot(DW[row]); axes[1].set_title(f"DW row {row}  (−d/dt bump, d=1)"); axes[1].set_xlabel("time index")
plt.tight_layout()
plt.show()

## 4 · TMI with Galerkin dynamics

Adds a sparse polynomial ODE constraint on $z$:

$$\mathbf{DW}\,z \;\approx\; \mathbf{W}\,\boldsymbol{\theta}(z)\,\boldsymbol{\xi}_z$$

$\boldsymbol{\xi}_z$ is discovered by **EnAdSR** (ensemble adaptive sparse
regression with SLIC model selection) and frozen to its active support
between sparse-regression steps.

In [ ]:
model_dyn = TMIWithDynamics(
    K           = K,
    W           = W,
    DW          = DW,
    N           = N,
    lambda_z    = 8.0,     # dynamics constraint weight
    sparse_iter = 3_000,   # run EnAdSR every 3k iters
    max_iters   = 12_000,
    lr          = 8e-4,
    tol         = 1e-5,
    normalize   = True,
    verbose     = True,
)
model_dyn.fit(X)

In [ ]:
q_dyn  = model_dyn.q_.numpy()
z_dyn  = model_dyn.z_.numpy().squeeze()
xi_z   = model_dyn.xi_z_.numpy()          # (L, K) sparse coefficients
z_dyn_trunc = z_dyn[:W.shape[0]]          # match windowed time axis

kld_dyn = float(np.mean(np.sum(X * np.log((X + 1e-12) / (q_dyn + 1e-12)), axis=0)))
print(f"KLD (TMI+dynamics): {kld_dyn:.6f}")
print(f"\nDiscovered xi_z (L={N.shape[0]} library terms, K={K}):")
term_labels = []
for row in N.astype(int):
    label = "·".join(f"z{k}^{e}" for k, e in enumerate(row) if e > 0) or "1"
    term_labels.append(label)
for lbl, coeff in zip(term_labels, xi_z[:, 0]):
    marker = " ◀" if abs(coeff) > 1e-10 else ""
    print(f"  {lbl:<10s}: {coeff:+.5f}{marker}")

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

def show(ax, mat, title):
    im = ax.imshow(mat, aspect="auto", origin="lower",
                   extent=[t[0], t[-1], x[0], x[-1]], cmap="hot",
                   vmin=0, vmax=X.max())
    plt.colorbar(im, ax=ax, shrink=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("time t"); ax.set_ylabel("space x")

show(fig.add_subplot(gs[0, 0]), X,       "Data  X")
show(fig.add_subplot(gs[0, 1]), q_basic, "TMI (core)  q")
show(fig.add_subplot(gs[0, 2]), q_dyn,   "TMI + dynamics  q")

# z trajectories
ax = fig.add_subplot(gs[1, :2])
true_z = 1.0 / (4.0 * D * t)

def align(z_arr):
    s = np.abs(z_arr).mean() / np.abs(true_z).mean()
    return s * true_z if np.corrcoef(z_arr, true_z)[0,1] > 0 else -s * true_z

ax.plot(t, z_basic, lw=2,   label="TMI z (core)")
ax.plot(t, z_dyn,   lw=2,   label="TMI z (dynamics)")
ax.plot(t, align(z_dyn), "--", lw=1.5, label="1/(4Dt) (scaled)", c="k")
ax.set_xlabel("time t"); ax.set_title("Temporal latent z(t)", fontsize=10)
ax.legend(fontsize=8)

# Residual DW @ z vs W @ theta @ xi_z
ax2 = fig.add_subplot(gs[1, 2])
z_t   = torch.tensor(model_dyn.z_.numpy(), dtype=torch.float64)
N_t   = torch.tensor(N,  dtype=torch.float64)
W_t   = torch.tensor(W,  dtype=torch.float64)
DW_t  = torch.tensor(DW, dtype=torch.float64)
xi_t  = model_dyn.xi_z_
from pydme.library import poly_library
with torch.no_grad():
    theta = poly_library(z_t, N_t)
    lhs   = (DW_t @ z_t).numpy()
    rhs   = (W_t  @ theta @ xi_t).numpy()
ax2.plot(lhs[:, 0], label="DW @ z", lw=1.5)
ax2.plot(rhs[:, 0], "--", label="W @ θ(z) @ ξ_z", lw=1.5)
ax2.set_title("Dynamics fit: DW·z ≈ W·θ(z)·ξz", fontsize=9)
ax2.set_xlabel("window index"); ax2.legend(fontsize=8)

plt.suptitle("TMI Gaussian Diffusion Demo", fontsize=12, y=1.01)
plt.show()

## 5 · Convergence curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

for ax, model, label in [
    (axes[0], model_basic, "TMI core"),
    (axes[1], model_dyn,   "TMI + dynamics"),
]:
    losses = model.losses_.numpy()
    grads  = model.grads_.numpy()
    iters  = np.arange(1, len(losses)+1)
    ax2    = ax.twinx()
    ax.semilogy(iters, losses, c="steelblue", lw=1.5, label="loss")
    ax2.semilogy(iters, grads,  c="tomato",    lw=1.2, alpha=0.7, label="‖grad‖")
    ax.set_xlabel("iteration")
    ax.set_ylabel("KLD loss", color="steelblue")
    ax2.set_ylabel("‖grad‖", color="tomato")
    ax.set_title(label, fontsize=10)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()